In [0]:
%pip install --upgrade mne

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
"""
=========================================================
 CLASIFICACION DE ETAPAS DE SUENO - Sleep-EDF Telemetry
 Canales: EEG Fpz-Cz, EEG Pz-Oz, EOG Horizontal, EMG Submental, 
 Pipeline: EDF -> epocas 30s -> features -> RandomForest
=========================================================
"""

import numpy as np
import pandas as pd
from pathlib import Path
from scipy import signal, stats
import mlflow

import mne
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import (
    cohen_kappa_score, f1_score, classification_report, confusion_matrix
)

In [0]:
experiment = mlflow.set_experiment("/model_jj")

In [0]:
mne.set_log_level("ERROR")

# ── CONFIGURACION ────────────────────────────────────────
#BASE = Path("../sleep-edf-database-expanded-1.0.0/sleep-telemetry")
BASE = Path("./Data_Raw/")
CANALES = ["EEG Fpz-Cz", "EEG Pz-Oz"]
EPOCA_SEG = 30.0          # duracion de la epoca en segundos
L_FREQ, H_FREQ = 0.3, 35.0
FUNDIR_N3_N4 = True       # AASM moderno: N4 se fusiona en N3
# Bandas de frecuencia clasicas del EEG de sueno (Hz)
BANDAS = {
    "delta": (0.5, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 12.0),
    "sigma": (12.0, 16.0),   # husos de sueno
    "beta":  (16.0, 30.0),
}

# Mapeo de etiquetas del hipnograma a clases
MAPA_ETAPAS = {
    "Sleep stage W": "W",
    "Sleep stage 1": "N1",
    "Sleep stage 2": "N2",
    "Sleep stage 3": "N3",
    "Sleep stage 4": "N4",
    "Sleep stage R": "REM",
}
ETAPAS_VALIDAS = ["W", "N1", "N2", "N3", "REM"] if FUNDIR_N3_N4 \
    else ["W", "N1", "N2", "N3", "N4", "REM"]

In [0]:
# =========================================================
# 1. CARGA Y PREPROCESADO
# =========================================================
def cargar_registro(psg_file, hyp_file, canales=CANALES):
    """Carga el PSG y el hipnograma, filtra y devuelve (raw, annotations)."""
    raw = mne.io.read_raw_edf(psg_file, preload=True)
    raw.pick(canales)

    # Pasa-banda: quita deriva DC lenta y ruido de alta frecuencia
    raw.filter(l_freq=L_FREQ, h_freq=H_FREQ, verbose=False)

    ann = mne.read_annotations(hyp_file)
    return raw, ann


def etiquetas_por_epoca(ann, n_epocas, epoca_seg=EPOCA_SEG):
    """
    Convierte las anotaciones (onset/duration/description) en un vector
    de etiquetas, una por epoca. Devuelve None donde no hay anotacion.
    """
    etiquetas = np.full(n_epocas, None, dtype=object)

    for onset, dur, desc in zip(ann.onset, ann.duration, ann.description):
        etapa = MAPA_ETAPAS.get(str(desc).strip())
        if etapa is None:
            continue  # ignora "Movement time", "Sleep stage ?", etc.
        if FUNDIR_N3_N4 and etapa == "N4":
            etapa = "N3"

        i_ini = int(np.floor(onset / epoca_seg))
        i_fin = int(np.ceil((onset + dur) / epoca_seg))
        i_ini, i_fin = max(0, i_ini), min(n_epocas, i_fin)
        etiquetas[i_ini:i_fin] = etapa

    return etiquetas


def segmentar_epocas(raw, epoca_seg=EPOCA_SEG):
    """
    Corta la senal continua en epocas de `epoca_seg` segundos.
    Devuelve array (n_epocas, n_canales, n_muestras) y la fs.
    """
    fs = raw.info["sfreq"]
    datos = raw.get_data() * 1e6          # a microvoltios
    n_canales, n_total = datos.shape

    muestras_epoca = int(round(epoca_seg * fs))
    n_epocas = n_total // muestras_epoca

    # Recorta el sobrante y reordena a (n_epocas, n_canales, n_muestras)
    datos = datos[:, :n_epocas * muestras_epoca]
    epocas = datos.reshape(n_canales, n_epocas, muestras_epoca)
    epocas = np.transpose(epocas, (1, 0, 2))

    return epocas, fs

In [0]:
# =========================================================
# 2. EXTRACCION DE FEATURES
# =========================================================
def parametros_hjorth(x):
    """Actividad, movilidad y complejidad de Hjorth."""
    dx = np.diff(x)
    ddx = np.diff(dx)

    var_x = np.var(x)
    var_dx = np.var(dx)
    var_ddx = np.var(ddx)

    actividad = var_x
    movilidad = np.sqrt(var_dx / var_x) if var_x > 0 else 0.0
    mov_dx = np.sqrt(var_ddx / var_dx) if var_dx > 0 else 0.0
    complejidad = mov_dx / movilidad if movilidad > 0 else 0.0

    return actividad, movilidad, complejidad


def features_canal(x, fs, prefijo):
    """Extrae el set de features de un canal para una epoca."""
    f = {}

    # --- Espectro via Welch (ventanas de 4s -> resolucion 0.25 Hz) ---
    nperseg = min(len(x), int(4 * fs))
    freqs, psd = signal.welch(x, fs=fs, nperseg=nperseg)

    # Potencia absoluta por banda (integral de la PSD)
    potencias = {}
    for nombre, (lo, hi) in BANDAS.items():
        mask = (freqs >= lo) & (freqs < hi)
        potencias[nombre] = np.trapz(psd[mask], freqs[mask]) if mask.any() else 0.0

    total = sum(potencias.values())

    # Potencia relativa (mas robusta entre sujetos que la absoluta)
    for nombre, p in potencias.items():
        f[f"{prefijo}_rel_{nombre}"] = p / total if total > 0 else 0.0
        f[f"{prefijo}_log_{nombre}"] = np.log10(p + 1e-12)

    # Ratios entre bandas: muy discriminativos entre etapas
    eps = 1e-12
    f[f"{prefijo}_ratio_delta_beta"] = potencias["delta"] / (potencias["beta"] + eps)
    f[f"{prefijo}_ratio_theta_alpha"] = potencias["theta"] / (potencias["alpha"] + eps)
    f[f"{prefijo}_ratio_delta_theta"] = potencias["delta"] / (potencias["theta"] + eps)
    f[f"{prefijo}_ratio_alpha_beta"] = potencias["alpha"] / (potencias["beta"] + eps)

    # Entropia espectral (baja en sueno profundo, alta en vigilia)
    psd_norm = psd / (psd.sum() + eps)
    f[f"{prefijo}_entropia_espectral"] = float(
        -np.sum(psd_norm * np.log2(psd_norm + eps))
    )

    # Frecuencia mediana y borde espectral al 95%
    psd_acum = np.cumsum(psd)
    if psd_acum[-1] > 0:
        psd_acum = psd_acum / psd_acum[-1]
        f[f"{prefijo}_freq_mediana"] = float(freqs[np.searchsorted(psd_acum, 0.50)])
        f[f"{prefijo}_borde_espectral_95"] = float(freqs[np.searchsorted(psd_acum, 0.95)])
    else:
        f[f"{prefijo}_freq_mediana"] = 0.0
        f[f"{prefijo}_borde_espectral_95"] = 0.0

    # --- Estadisticas en el dominio del tiempo ---
    f[f"{prefijo}_std"] = float(np.std(x))
    f[f"{prefijo}_ptp"] = float(np.ptp(x))                  # pico a pico
    f[f"{prefijo}_kurtosis"] = float(stats.kurtosis(x))
    f[f"{prefijo}_skew"] = float(stats.skew(x))
    f[f"{prefijo}_percentil_75"] = float(np.percentile(np.abs(x), 75))

    # Tasa de cruces por cero (proxy de frecuencia dominante)
    f[f"{prefijo}_zcr"] = float(np.mean(np.diff(np.signbit(x)) != 0))

    # Parametros de Hjorth
    act, mov, comp = parametros_hjorth(x)
    f[f"{prefijo}_hjorth_actividad"] = float(act)
    f[f"{prefijo}_hjorth_movilidad"] = float(mov)
    f[f"{prefijo}_hjorth_complejidad"] = float(comp)

    return f


def extraer_features(epocas, fs, nombres_canales=CANALES):
    """Construye el DataFrame de features (una fila por epoca)."""
    filas = []
    for i in range(epocas.shape[0]):
        fila = {}
        for c, nombre in enumerate(nombres_canales):
            prefijo = nombre.replace("EEG ", "").replace("-", "")
            fila.update(features_canal(epocas[i, c, :], fs, prefijo))
        filas.append(fila)

    return pd.DataFrame(filas)


def agregar_contexto(X, n_vecinos=1):
    """
    Concatena las features de las epocas vecinas (t-n ... t+n).
    Aprovecha que las etapas de sueno tienen fuerte estructura temporal.
    """
    partes = [X.add_suffix("_t0")]
    for k in range(1, n_vecinos + 1):
        partes.append(X.shift(k).add_suffix(f"_t-{k}"))
        partes.append(X.shift(-k).add_suffix(f"_t+{k}"))

    X_ctx = pd.concat(partes, axis=1)
    # Rellena los bordes (primeras/ultimas epocas) con la epoca actual
    return X_ctx.bfill().ffill()

In [0]:
# =========================================================
# 3. PIPELINE POR SUJETO
# =========================================================
def procesar_sujeto(psg_file, hyp_file, sujeto_id, n_vecinos=1):
    """Devuelve (X, y, grupos) listos para el modelo, para un sujeto."""
    raw, ann = cargar_registro(psg_file, hyp_file)
    epocas, fs = segmentar_epocas(raw)
    y = etiquetas_por_epoca(ann, n_epocas=epocas.shape[0])

    X = extraer_features(epocas, fs)
    X = agregar_contexto(X, n_vecinos=n_vecinos)

    # Descarta epocas sin etiqueta valida
    valido = np.array([e in ETAPAS_VALIDAS for e in y])
    X, y = X[valido].reset_index(drop=True), y[valido]

    grupos = np.full(len(y), sujeto_id)
    print(f"  {sujeto_id}: {len(y)} epocas validas de {epocas.shape[0]}")

    return X, pd.Series(y, name="etapa"), grupos


def construir_dataset(pares_archivos, n_vecinos=1):
    """
    pares_archivos: lista de tuplas (psg_file, hyp_file, sujeto_id).
    Devuelve X, y, grupos concatenados de todos los sujetos.
    """
    Xs, ys, gs = [], [], []
    for psg, hyp, sid in pares_archivos:
        X, y, g = procesar_sujeto(psg, hyp, sid, n_vecinos=n_vecinos)
        Xs.append(X)
        ys.append(y)
        gs.append(g)

    X = pd.concat(Xs, ignore_index=True)
    y = pd.concat(ys, ignore_index=True)
    grupos = np.concatenate(gs)

    return X, y, grupos

In [0]:
# =========================================================
# 4. ENTRENAMIENTO Y EVALUACION
# =========================================================
def evaluar(y_true, y_pred, etiquetas=None):
    """Imprime las metricas estandar en sleep staging."""
    etiquetas = etiquetas or [e for e in ETAPAS_VALIDAS if e in set(y_true)]

    kappa = cohen_kappa_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    acc = (np.asarray(y_true) == np.asarray(y_pred)).mean()

    print(f"\n  Accuracy      : {acc:.3f}")
    mlflow.log_metric("Accuracy", acc)
    print(f"  Cohen's kappa : {kappa:.3f}   <- la metrica de referencia")
    mlflow.log_metric("Cohen's kappa ", kappa)
    print(f"  F1 macro      : {f1:.3f}\n")
    mlflow.log_metric("F1 macro", f1)

    print(classification_report(y_true, y_pred, zero_division=0))

    cm = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=etiquetas),
        index=[f"real_{e}" for e in etiquetas],
        columns=[f"pred_{e}" for e in etiquetas],
    )
    print("Matriz de confusion:")
    print(cm)

    return dict(accuracy=acc, kappa=kappa, f1_macro=f1, confusion=cm)

def importancia_features(modelo, X, top=20):
    """Devuelve las features mas importantes del RandomForest."""
    imp = pd.Series(modelo.feature_importances_, index=X.columns)
    return imp.sort_values(ascending=False).head(top)


In [0]:
def entrenar_rf(X, y, grupos=None, n_splits=5, semilla=42):
    """
    Entrena un RandomForest. Si hay >1 sujeto usa GroupKFold
    (division POR SUJETO, nunca aleatoria por epoca).
    """
    n_estimators = 300
    max_depth = None
    min_samples_leaf = 2
    max_features = "sqrt"
    criterion = "gini"

    modelo = RandomForestClassifier(
        criterion=criterion,
        n_estimators=n_estimators,
        max_depth=None,
        min_samples_leaf=2,
        max_features=max_features,
        bootstrap=True,
        oob_score=True, 
        class_weight="balanced_subsample",  # compensa el desbalance (N1 es raro)
        n_jobs=-1,
        random_state=semilla,
    )

    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("min_samples_leaf", min_samples_leaf)
    mlflow.log_param("max_features", max_features)
    mlflow.log_param("criterion", criterion)

    n_sujetos = len(np.unique(grupos)) if grupos is not None else 1

    if n_sujetos >= 2:
        n_splits = min(n_splits, n_sujetos)
        print(f"\nValidacion cruzada GroupKFold ({n_splits} folds, por sujeto)")
        gkf = GroupKFold(n_splits=n_splits)

        y_true_all, y_pred_all = [], []
        for fold, (i_tr, i_te) in enumerate(gkf.split(X, y, groups=grupos), 1):
            modelo.fit(X.iloc[i_tr], y.iloc[i_tr])
            pred = modelo.predict(X.iloc[i_te])
            k = cohen_kappa_score(y.iloc[i_te], pred)
            print(f"  Fold {fold}: kappa = {k:.3f}")
            y_true_all.extend(y.iloc[i_te])
            y_pred_all.extend(pred)

        print("\n--- Resultado agregado (todos los folds) ---")
        metricas = evaluar(y_true_all, y_pred_all)
    else:
        # Un solo sujeto: division temporal (no aleatoria) para no
        # filtrar informacion entre epocas contiguas
        print("\nUn solo sujeto: division temporal 70/30 (sin shuffle)")
        corte = int(len(X) * 0.7)
        X_tr, X_te = X.iloc[:corte], X.iloc[corte:]
        y_tr, y_te = y.iloc[:corte], y.iloc[corte:]

        modelo.fit(X_tr, y_tr)
        metricas = evaluar(y_te, modelo.predict(X_te))

    # Reentrena con todo para el modelo final
    modelo.fit(X, y)
    mlflow.sklearn.log_model(modelo, name="random-forest-model")

    return modelo, metricas

In [0]:
with mlflow.start_run(experiment_id=experiment.experiment_id):
    mlflow.autolog()
    mlflow.log_param("EPOCA_SEG", EPOCA_SEG)
    mlflow.log_param("L_FREQ", L_FREQ)
    mlflow.log_param("H_FREQ", H_FREQ)
    mlflow.log_param("FUNDIR_N3_N4", FUNDIR_N3_N4)
    mlflow.log_param("BANDAS", BANDAS)
    # Un solo sujeto (tu caso actual)
    archivos = [
        (BASE / "ST7011J0-PSG.edf", BASE / "ST7011JP-Hypnogram.edf", "ST7011"),
        (BASE / "ST7021J0-PSG.edf", BASE / "ST7021JM-Hypnogram.edf", "ST7021"),
        (BASE / "ST7022J0-PSG.edf", BASE / "ST7022JM-Hypnogram.edf", "ST7022"),
    ]

    print("Construyendo dataset...")
    X, y, grupos = construir_dataset(archivos, n_vecinos=1)

    print(f"\nDataset: {X.shape[0]} epocas x {X.shape[1]} features")
    print("\nDistribucion de clases:")
    print(y.value_counts().sort_index())

    modelo, metricas = entrenar_rf(X, y, grupos)

    print("\n--- Top 20 features mas importantes ---")
    print(importancia_features(modelo, X, top=20))

mlflow.end_run()

2026/08/26 05:36:08 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


Construyendo dataset...


/home/spark-b663a80a-9752-4ca1-9f8d-a3/.ipykernel/89/command-8879400230644202-1941518193:33: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  potencias[nombre] = np.trapz(psd[mask], freqs[mask]) if mask.any() else 0.0
/home/spark-b663a80a-9752-4ca1-9f8d-a3/.ipykernel/89/command-8879400230644202-1941518193:68: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  f[f"{prefijo}_kurtosis"] = float(stats.kurtosis(x))
/home/spark-b663a80a-9752-4ca1-9f8d-a3/.ipykernel/89/command-8879400230644202-1941518193:69: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  f[f"{prefijo}_skew"] = float(stats.skew(x))


  ST7011: 1092 epocas validas de 1196


/home/spark-b663a80a-9752-4ca1-9f8d-a3/.ipykernel/89/command-8879400230644202-1941518193:33: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  potencias[nombre] = np.trapz(psd[mask], freqs[mask]) if mask.any() else 0.0
/home/spark-b663a80a-9752-4ca1-9f8d-a3/.ipykernel/89/command-8879400230644202-1941518193:68: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  f[f"{prefijo}_kurtosis"] = float(stats.kurtosis(x))
/home/spark-b663a80a-9752-4ca1-9f8d-a3/.ipykernel/89/command-8879400230644202-1941518193:69: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  f[f"{prefijo}_skew"] = float(stats.skew(x))


...

  Fold 2: kappa = 0.785


🔗 View Logged Model at: https://dbc-d9e602d0-28c6.cloud.databricks.com/ml/experiments/464757408341871/models/m-2cdee6eda39f4bfab1b65ef873c09c37?o=7474646184403256


  Fold 3: kappa = 0.761

--- Resultado agregado (todos los folds) ---

  Accuracy      : 0.704
  Cohen's kappa : 0.603   <- la metrica de referencia
  F1 macro      : 0.678

              precision    recall  f1-score   support

          N1       0.46      0.47      0.47       219
          N2       0.81      0.65      0.72      1294
          N3       0.94      0.82      0.88       639
         REM       0.43      0.73      0.54       457
           W       0.77      0.78      0.78       347

    accuracy                           0.70      2956
   macro avg       0.68      0.69      0.68      2956
weighted avg       0.75      0.70      0.72      2956

Matriz de confusion:
          pred_W  pred_N1  pred_N2  pred_N3  pred_REM
real_W       271       59        4        0        13
real_N1       44      103       28        2        42
real_N2       23       26      847       33       365
real_N3        3        0       89      527        20
real_REM       9       35       79        0   

🔗 View Logged Model at: https://dbc-d9e602d0-28c6.cloud.databricks.com/ml/experiments/464757408341871/models/m-f739f7e5d8384c5b972921ef5b363374?o=7474646184403256
🔗 View Logged Model at: https://dbc-d9e602d0-28c6.cloud.databricks.com/ml/experiments/464757408341871/models/m-de3a320477f84514bbcdfc2998b07092?o=7474646184403256
2026/08/26 05:37:10 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.



--- Top 20 features mas importantes ---
FpzCz_rel_beta_t0             0.029103
PzOz_rel_beta_t0              0.028793
PzOz_ratio_alpha_beta_t0      0.028544
FpzCz_ratio_alpha_beta_t0     0.025751
PzOz_rel_beta_t+1             0.025025
PzOz_ratio_theta_alpha_t0     0.024810
PzOz_ratio_delta_beta_t0      0.024426
FpzCz_ratio_delta_beta_t0     0.021962
FpzCz_log_sigma_t0            0.020508
FpzCz_log_sigma_t+1           0.017986
FpzCz_log_delta_t0            0.017782
PzOz_ratio_alpha_beta_t-1     0.017232
PzOz_ratio_alpha_beta_t+1     0.016275
FpzCz_ratio_alpha_beta_t-1    0.015769
FpzCz_rel_beta_t-1            0.015383
PzOz_ratio_delta_beta_t+1     0.014860
FpzCz_rel_beta_t+1            0.013356
PzOz_rel_beta_t-1             0.013346
PzOz_ratio_delta_beta_t-1     0.013151
PzOz_log_beta_t0              0.012448
dtype: float64
